In [97]:
import pandas as pd
import geopandas as gpd
import geopandas as gpd
from shapely.geometry import Point

In [98]:
left_bank = [
    "Śródmieście", "Wola", "Ochota", "Mokotów",
    "Żoliborz", "Bemowo", "Ursynów", "Włochy",
    "Bielany", "Ursus", "Wilanów"
]

right_bank = [
    "Praga-Północ", "Praga-Południe", "Targówek",
    "Białołęka", "Rembertów", "Wawer", "Wesoła"
]

In [99]:
def get_bank_label(district_name):
    if pd.isna(district_name):
        return "Outer"
        
    d_clean = str(district_name).replace("-", " ").strip()
    
    left_clean = [d.replace("-", " ") for d in left_bank]
    right_clean = [d.replace("-", " ") for d in right_bank]

    if d_clean in left_clean:
        return "Left"
    elif d_clean in right_clean:
        return "Right"
    else:
        return "Outer"

In [100]:

class WarsawRiverBankClassifier:
    def __init__(self, geojson_path):

        warsaw_map = gpd.read_file(geojson_path)
        warsaw_map = warsaw_map.to_crs(epsg=4326)

        self.districts_list = [
            (row['geometry'], row['name']) 
            for i, row in warsaw_map.iterrows()
            if str(row['name']).strip() != 'Warszawa'
        ]

    def classify_point(self, lat, lon):
        if pd.isna(lat) or pd.isna(lon):
            return "Outer"
            
        punkt = Point(lon, lat)
        
        for d_geom, d_name in self.districts_list:
            if d_geom.contains(punkt):
                return get_bank_label(d_name)
                
        return "Outer"


    
   

In [101]:
df=pd.read_csv('warsaw_osm_network_stream.csv')

In [102]:

GEOJSON_FILE = "warszawa-dzielnice.geojson"
classifier = WarsawRiverBankClassifier(GEOJSON_FILE)

df['pickup_d'] = df.apply( lambda row: classifier.classify_point(row['start_lat'], row['start_lon']), axis=1 )
    
df['dropoff_d'] = df.apply(lambda row: classifier.classify_point(row['end_lat'], row['end_lon']), axis=1)


In [103]:
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values('timestamp')

df["hour"] = df["timestamp"].dt.hour

df["is_weekend"] = df["is_weekend"].astype(int)
df["rush_hour"] = df["rush_hour"].astype(int)
df["is_holiday"] = df["is_holiday"].astype(int)

In [104]:
df.head(5)

,timestamp,start_lat,start_lon,end_lat,end_lon,distance_km,travel_time_min,hour,day_of_week,is_weekend,rush_hour,weather,is_holiday,pickup_d,dropoff_d
0,2025-01-01 00:09:06,52.253264,20.981510,52.159729,21.126328,17.786888,25.131083,0,2,0,0,clear,1,Left,Left
1,2025-01-01 00:17:26,52.191368,21.043104,52.145980,21.037312,6.231378,11.619191,0,2,0,0,rain,1,Left,Left
2,2025-01-01 00:34:09,52.267529,21.032690,52.149947,21.053613,17.694589,55.764620,0,2,0,0,snow,1,Right,Left
3,2025-01-01 00:35:16,52.177431,21.062454,52.218237,20.999811,7.952929,19.220696,0,2,0,0,fog,1,Left,Left
4,2025-01-01 00:42:18,52.253970,21.220869,52.196762,20.895023,28.631239,42.313451,0,2,0,0,rain,1,Right,Left


In [105]:
categorical_cols = ["weather"]

df = pd.get_dummies(df, columns=categorical_cols, prefix=categorical_cols)

In [106]:
df["weather_clear"] = df["weather_clear"].astype(int)
df["weather_fog"] = df["weather_fog"].astype(int)
df["weather_rain"] = df["weather_rain"].astype(int)
df["weather_snow"] = df["weather_snow"].astype(int)


In [107]:
df.head(5)

,timestamp,start_lat,start_lon,end_lat,end_lon,distance_km,travel_time_min,hour,day_of_week,is_weekend,rush_hour,is_holiday,pickup_d,dropoff_d,weather_clear,weather_fog,weather_rain,weather_snow
0,2025-01-01 00:09:06,52.253264,20.981510,52.159729,21.126328,17.786888,25.131083,0,2,0,0,1,Left,Left,1,0,0,0
1,2025-01-01 00:17:26,52.191368,21.043104,52.145980,21.037312,6.231378,11.619191,0,2,0,0,1,Left,Left,0,0,1,0
2,2025-01-01 00:34:09,52.267529,21.032690,52.149947,21.053613,17.694589,55.764620,0,2,0,0,1,Right,Left,0,0,0,1
3,2025-01-01 00:35:16,52.177431,21.062454,52.218237,20.999811,7.952929,19.220696,0,2,0,0,1,Left,Left,0,1,0,0
4,2025-01-01 00:42:18,52.253970,21.220869,52.196762,20.895023,28.631239,42.313451,0,2,0,0,1,Right,Left,0,0,1,0


In [109]:
df["pickup_d"].unique()

<ArrowStringArray>
['Left', 'Right', 'Outer']
Length: 3, dtype: str

In [108]:
df.to_csv('warsaw_synthetic.csv', index=False, header=True)
